# Homework 3

For this HA you should return the .ipynb file name in the form `HA_3_Surname_Name.ipynb`. Each solution is a function with name `prob_i`, where `i` is the number of the problem. Make imports ONLY in the cell below. Remember, despite the fact that inputs for your function are `DataFrames`, you need to solve ***all*** problems *fully* in SQL and output a pandas `DataFrame` as a result

## Imports

In [11]:
import pandas as pd
import os
import psycopg2

## Problem 1 (50 pts)

Table: `Employee`


| Column Name  | Type    |
|--------------|---------|
| id           | int     |
| name         | varchar |
| salary       | int     |
| departmentId | int     |

id is the primary key (column with unique values) for this table.\
departmentId is a foreign key (reference column) of the ID from the Department table.\
Each row of this table indicates the ID, name, and salary of an employee. It also contains the ID of their department.
 

Table: `Department`


| Column Name | Type    |
|-------------|---------|
| id          | int     |
| name        | varchar |

id is the primary key (column with unique values) for this table.\
Each row of this table indicates the ID of a department and its name.
 

A company's executives are interested in seeing who earns the most money in each of the company's departments. A high earner in a department is an employee who has a salary in the top three unique salaries for that department.

Write a solution to find the employees who are high earners in each of the departments.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: \
Employee table:

| id | name  | salary | departmentId |
|----|-------|--------|--------------|
| 1  | Joe   | 85000  | 1            |
| 2  | Henry | 80000  | 2            |
| 3  | Sam   | 60000  | 2            |
| 4  | Max   | 90000  | 1            |
| 5  | Janet | 69000  | 1            |
| 6  | Randy | 85000  | 1            |
| 7  | Will  | 70000  | 1            |

Department table:

| id | name  |
|----|-------|
| 1  | IT    |
| 2  | Sales |

Output: 

| Department | Employee | Salary |
|------------|----------|--------|
| IT         | Max      | 90000  |
| IT         | Joe      | 85000  |
| IT         | Randy    | 85000  |
| IT         | Will     | 70000  |
| Sales      | Henry    | 80000  |
| Sales      | Sam      | 60000  |

Explanation: \
In the IT department:
- Max earns the highest unique salary
- Both Randy and Joe earn the second-highest unique salary
- Will earns the third-highest unique salary

In the Sales department:
- Henry earns the highest salary
- Sam earns the second-highest salary
- There is no third-highest salary as there are only two employees
 

Constraints:

There are no employees with the exact same name, salary and department.

*Hint: for this the only new function you'll need is `DENSE_RANK()`, you also should use the `WITH` keyword*

In [3]:
def top_three_salaries(employee: pd.DataFrame, department: pd.DataFrame) -> pd.DataFrame:
    ###your code here
    return

In [43]:
path = os.path.join('.', 'tests', 'Problem_1')

files = sorted(os.listdir(path))

emps = map(lambda x: pd.read_csv(os.path.join(path, x), index_col=0), filter(lambda x: 'emp' in x, files))
deps = map(lambda x: pd.read_csv(os.path.join(path, x), index_col=0), filter(lambda x: 'dep' in x, files))
exps = map(lambda x: pd.read_csv(os.path.join(path, x), index_col=0), filter(lambda x: 'exp' in x, files))


for idx, (emp, dep, exp) in enumerate(zip(emps, deps, exps)):
    out = top_three_salaries(emp, dep).sort_values('Salary').reset_index(drop=True).astype(exp.dtypes)
    exp = exp.sort_values('Salary').reset_index(drop=True)
    try:
        assert out.equals(exp)
    except:
        print(f'Error in case {idx}!\nInputs:\nEmployees:\n{emp}\n\nDepartments:\n{dep}\n\nExpected:\n{exp}\n\nGot:\n{out}')
        break
else:
    print('All is correct!')

All is correct!


## Problem 2 (50 pts)

Table: `Trips`


| Column Name | Type     |
|-------------|----------|
| id          | int      |
| client_id   | int      |
| driver_id   | int      |
| city_id     | int      |
| status      | enum     |
| request_at  | varchar  |     

id is the primary key (column with unique values) for this table.\
The table holds all taxi trips. Each trip has a unique id, while client_id and driver_id are foreign keys to the users_id at the Users table.\
Status is an ENUM (category) type of ('completed', 'cancelled_by_driver', 'cancelled_by_client').

Table: Users


| Column Name | Type     |
|-------------|----------|
| users_id    | int      |
| banned      | bool     |
| role        | enum     |

users_id is the primary key (column with unique values) for this table.\
The table holds all users. Each user has a unique users_id, and role is an ENUM type of ('client', 'driver', 'partner').\
banned is an ENUM (category) type of ('Yes', 'No').

The cancellation rate is computed by dividing the number of canceled (by client or driver) requests with unbanned users by the total number of requests with unbanned users on that day.

Write a solution to find the cancellation rate of requests with unbanned users (both client and driver must not be banned) each day between "2013-10-01" and "2013-10-03" with at least one trip. Round Cancellation Rate to two decimal points.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: \
Trips table:

| id | client_id | driver_id | city_id | status              | request_at |
|----|-----------|-----------|---------|---------------------|------------|
| 1  | 1         | 10        | 1       | completed           | 2013-10-01 |
| 2  | 2         | 11        | 1       | cancelled_by_driver | 2013-10-01 |
| 3  | 3         | 12        | 6       | completed           | 2013-10-01 |
| 4  | 4         | 13        | 6       | cancelled_by_client | 2013-10-01 |
| 5  | 1         | 10        | 1       | completed           | 2013-10-02 |
| 6  | 2         | 11        | 6       | completed           | 2013-10-02 |
| 7  | 3         | 12        | 6       | completed           | 2013-10-02 |
| 8  | 2         | 12        | 12      | completed           | 2013-10-03 |
| 9  | 3         | 10        | 12      | completed           | 2013-10-03 |
| 10 | 4         | 13        | 12      | cancelled_by_driver | 2013-10-03 |

Users table:

| users_id | banned | role   |
|----------|--------|--------|
| 1        | No     | client |
| 2        | Yes    | client |
| 3        | No     | client |
| 4        | No     | client |
| 10       | No     | driver |
| 11       | No     | driver |
| 12       | No     | driver |
| 13       | No     | driver |

Output: 

| Day        | Cancellation Rate |
|------------|-------------------|
| 2013-10-01 | 0.33              |
| 2013-10-02 | 0.00              |
| 2013-10-03 | 0.50              |

Explanation: \
On 2013-10-01:
  - There were 4 requests in total, 2 of which were canceled.
  - However, the request with Id=2 was made by a banned client (User_Id=2), so it is ignored in the calculation.
  - Hence there are 3 unbanned requests in total, 1 of which was canceled.
  - The Cancellation Rate is (1 / 3) = 0.33
On 2013-10-02:
  - There were 3 requests in total, 0 of which were canceled.
  - The request with Id=6 was made by a banned client, so it is ignored.
  - Hence there are 2 unbanned requests in total, 0 of which were canceled.
  - The Cancellation Rate is (0 / 2) = 0.00
On 2013-10-03:
  - There were 3 requests in total, 1 of which was canceled.
  - The request with Id=8 was made by a banned client, so it is ignored.
  - Hence there are 2 unbanned request in total, 1 of which were canceled.
  - The Cancellation Rate is (1 / 2) = 0.50

In [32]:
def trips_and_users(trips: pd.DataFrame, users: pd.DataFrame) -> pd.DataFrame:
    ###your code here
    return

In [42]:
path = os.path.join('.', 'tests', 'Problem_2')

files = sorted(os.listdir(path))

trips = map(lambda x: pd.read_csv(os.path.join(path, x), index_col=0), filter(lambda x: 'trip' in x, files))
users = map(lambda x: pd.read_csv(os.path.join(path, x), index_col=0), filter(lambda x: 'user' in x, files))
exps = map(lambda x: pd.read_csv(os.path.join(path, x), index_col=0), filter(lambda x: 'exp' in x, files))

for idx, (trip, user, exp) in enumerate(zip(trips, users, exps)):
    out = trips_and_users(trip, user).sort_values('Day').reset_index(drop=True).astype(exp.dtypes)
    exp = exp.sort_values('Day').reset_index(drop=True)
    try:
        assert out.equals(exp)
    except:
        print(f'Error in case {idx}!\nInputs:\nTrips:\n{trip}\n\nUsers:\n{user}\n\nExpected:\n{exp}\n\nGot:\n{out}')
        break
else:
    print('All is correct!')

All is correct!
